# CL39 attention/confidence analysis playground

Run this notebook with `diffusion_template/` as the working directory. The Serv trainer/YAML runs are the reproducible source of truth; this notebook imports the rendering helpers for quick exploration of their downloaded artifacts.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from tools.analysis.analyze_cl39_attention import (
    PROJECT_ROOT, DEFAULT_CHECKPOINT_DIR, _weighted_map,
    load_manifest, verify_checkpoint,
)

SERV_OUTPUT_ROOT = PROJECT_ROOT / 'artifacts' / 'cl39_attention_24k_serv_a100'
verify_checkpoint(DEFAULT_CHECKPOINT_DIR)

In [ ]:
records = load_manifest(SERV_OUTPUT_ROOT)
summary = json.loads((SERV_OUTPUT_ROOT / 'summary.json').read_text())
per_sample = pd.read_csv(SERV_OUTPUT_ROOT / 'per_sample_summary.csv')
display(pd.Series(summary).head(20))
per_sample[['index', 'identity', 'action', 'confidence_face', 'correction_to_native_face', 'actual', 'c1', 'ba_off']]

In [ ]:
record = records[0]
npz_path = SERV_OUTPUT_ROOT / 'telemetry' / f"{record['index']:02d}.npz"
with np.load(npz_path) as telemetry:
    confidence = _weighted_map(telemetry, 'confidence')
    correction = _weighted_map(telemetry, 'routed_delta_magnitude')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(confidence, vmin=.25, vmax=1, cmap='viridis'); axes[0].set_title('Confidence C')
axes[1].imshow(correction, cmap='magma'); axes[1].set_title('|applied correction|')
for ax in axes: ax.axis('off')

Publication-quality regeneration must use the three `CL39_attention_audit_serv_*.yaml` validation-only configs on Serv so the original 12-item batching is preserved. The render command below is safe for local figure iteration.

In [ ]:
# %run tools/analysis/analyze_cl39_attention.py render --output-root artifacts/cl39_attention_24k_serv_a100 --figure-dir analysis/assets/cl39_attention_24k_serv_a100 --skip-id-score